# Query 8: Cumulative Daily Revenue
**Type:** Window Function – Cumulative Sum
**Problem:** Calculate the running total of daily revenue for January 2015.

In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName('Q8_CumulativeRevenue') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions','8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 18:23:29 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 18:23:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 18:23:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/25 18:23:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/25 18:23:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 3.5.1


In [2]:
DATA_PATH = '../data/yellow_tripdata_2015-01.csv'

df = spark.read.option('header','true').option('inferSchema','true').csv(DATA_PATH)
df = df.withColumnRenamed('total_amount','total') \
       .withColumn('trip_date', F.to_date('tpep_pickup_datetime'))

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows:', df.count())

Total rows: 11157879


## RDD Implementation

In [3]:
start = time.time()

daily = (
    rdd
    .filter(lambda r: r['tpep_pickup_datetime'] is not None and r['total'] is not None)
    .map(lambda r: (str(r['tpep_pickup_datetime'])[:10], float(r['total'])))
    .reduceByKey(lambda a,b: a+b)
    .sortByKey()
    .collect()
)

cumulative, running = [], 0.0
for date, rev in daily:
    running += rev
    cumulative.append((date, round(rev,2), round(running,2)))

rdd_time = time.time() - start
print(f'RDD | Days: {len(cumulative)} | Time: {rdd_time:.2f}s')
print(f'{"Date":<12} {"Daily Rev":>14} {"Cumulative":>16}')
for d,r,c in cumulative[:10]:
    print(f'{d:<12} ${r:>13,.2f} ${c:>15,.2f}')

RDD | Days: 31 | Time: 70.61s
Date              Daily Rev       Cumulative
2015-01-01   $ 5,144,757.52 $   5,144,757.52
2015-01-02   $ 4,375,159.66 $   9,519,917.18
2015-01-03   $ 4,856,996.62 $  14,376,913.80
2015-01-04   $ 4,267,488.45 $  18,644,402.25
2015-01-05   $ 4,852,633.61 $  23,497,035.86
2015-01-06   $ 5,306,032.76 $  28,803,068.62
2015-01-07   $ 5,720,459.84 $  34,523,528.46
2015-01-08   $ 5,835,574.66 $  40,359,103.12
2015-01-09   $ 5,788,370.82 $  46,147,473.94
2015-01-10   $ 6,021,314.93 $  52,168,788.87


## DataFrame Implementation

In [4]:
start = time.time()

daily_df = (
    df.filter(F.col('total').isNotNull())
      .groupBy('trip_date')
      .agg(F.round(F.sum('total'),2).alias('daily_revenue'))
)

window_spec = Window.orderBy('trip_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)

result_df = (
    daily_df
    .withColumn('cumulative_revenue', F.round(F.sum('daily_revenue').over(window_spec),2))
    .orderBy('trip_date')
)
result_df.explain(True)
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show(15)

== Parsed Logical Plan ==
'Sort ['trip_date ASC NULLS FIRST], true
+- Project [trip_date#76, daily_revenue#162, cumulative_revenue#166]
   +- Project [trip_date#76, daily_revenue#162, _we0#167, round(_we0#167, 2) AS cumulative_revenue#166]
      +- Window [sum(daily_revenue#162) windowspecdefinition(trip_date#76 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS _we0#167], [trip_date#76 ASC NULLS FIRST]
         +- Project [trip_date#76, daily_revenue#162]
            +- Aggregate [trip_date#76], [trip_date#76, round(sum(total#55), 2) AS daily_revenue#162]
               +- Filter isnotnull(total#55)
                  +- Project [VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare_amount#29, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_

+----------+-------------+------------------+
| trip_date|daily_revenue|cumulative_revenue|
+----------+-------------+------------------+
|2015-01-01|   5144757.52|        5144757.52|
|2015-01-02|   4375159.66|        9519917.18|
|2015-01-03|   4856996.62|      1.43769138E7|
|2015-01-04|   4267488.45|     1.864440225E7|
|2015-01-05|   4852633.61|     2.349703586E7|
|2015-01-06|   5306032.76|     2.880306862E7|
|2015-01-07|   5720459.84|     3.452352846E7|
|2015-01-08|   5835574.66|     4.035910312E7|
|2015-01-09|   5788370.82|     4.614747394E7|
|2015-01-10|   6021314.93|     5.216878887E7|
|2015-01-11|   5259363.24|     5.742815211E7|
|2015-01-12|   4931473.23|     6.235962534E7|
|2015-01-13|   5963312.34|     6.832293768E7|
|2015-01-14|   6040182.11|     7.436311979E7|
|2015-01-15|   6232236.51|      8.05953563E7|
+----------+-------------+------------------+
only showing top 15 rows



## Spark SQL Implementation

In [7]:
start = time.time()

result_sql = spark.sql("""
    WITH daily AS (
        SELECT DATE(tpep_pickup_datetime)  AS trip_date,
               ROUND(SUM(total),2) AS daily_revenue
        FROM   trips
        WHERE  total IS NOT NULL
        GROUP BY DATE(tpep_pickup_datetime)
    )
    SELECT trip_date,
           daily_revenue,
           ROUND(SUM(daily_revenue) OVER (
               ORDER BY trip_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ),2) AS cumulative_revenue
    FROM   daily
    ORDER BY trip_date
""")
result_sql.explain(True)
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show(15)

== Parsed Logical Plan ==
CTE [daily]
:  +- 'SubqueryAlias daily
:     +- 'Aggregate ['DATE('tpep_pickup_datetime)], ['DATE('tpep_pickup_datetime) AS trip_date#200, 'ROUND('SUM('total), 2) AS daily_revenue#201]
:        +- 'Filter isnotnull('total)
:           +- 'UnresolvedRelation [trips], [], false
+- 'Sort ['trip_date ASC NULLS FIRST], true
   +- 'Project ['trip_date, 'daily_revenue, 'ROUND('SUM('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())), 2) AS cumulative_revenue#199]
      +- 'UnresolvedRelation [daily], [], false

== Analyzed Logical Plan ==
trip_date: date, daily_revenue: double, cumulative_revenue: double
WithCTE
:- CTERelationDef 2, false
:  +- SubqueryAlias daily
:     +- Aggregate [cast(tpep_pickup_datetime#18 as date)], [cast(tpep_pickup_datetime#18 as date) AS trip_date#200, round(sum(total#55), 2) AS daily_revenue#201]
:        +- Filter isnotnull(total#55)
:           +- SubqueryA

+----------+-------------+------------------+
| trip_date|daily_revenue|cumulative_revenue|
+----------+-------------+------------------+
|2015-01-01|   5144757.52|        5144757.52|
|2015-01-02|   4375159.66|        9519917.18|
|2015-01-03|   4856996.62|      1.43769138E7|
|2015-01-04|   4267488.45|     1.864440225E7|
|2015-01-05|   4852633.61|     2.349703586E7|
|2015-01-06|   5306032.76|     2.880306862E7|
|2015-01-07|   5720459.84|     3.452352846E7|
|2015-01-08|   5835574.66|     4.035910312E7|
|2015-01-09|   5788370.82|     4.614747394E7|
|2015-01-10|   6021314.93|     5.216878887E7|
|2015-01-11|   5259363.24|     5.742815211E7|
|2015-01-12|   4931473.23|     6.235962534E7|
|2015-01-13|   5963312.34|     6.832293768E7|
|2015-01-14|   6040182.11|     7.436311979E7|
|2015-01-15|   6232236.51|      8.05953563E7|
+----------+-------------+------------------+
only showing top 15 rows



## Performance Comparison

In [8]:
print('='*65)
print(f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}')
print('-'*65)
print(f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s')
print(f'{"Window Function":<25} {"Manual":>12} {"Built-in":>12} {"Built-in":>12}')
print(f'{"Distributed":<25} {"No":>12} {"Yes":>12} {"Yes":>12}')
print('='*65)
print('KEY INSIGHT: RDD collects to driver for cumulative sum.')
print('DataFrame/SQL use Tungsten Window operator natively.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                  70.61s        0.28s        0.92s
Window Function                 Manual     Built-in     Built-in
Distributed                         No          Yes          Yes
KEY INSIGHT: RDD collects to driver for cumulative sum.
DataFrame/SQL use Tungsten Window operator natively.
